<a href="https://colab.research.google.com/github/DataInfamous/pediatric-cancer-stalled-trials/blob/main/Pediatric_cancer_trial_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json

base_url = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "query.term": "pediatric cancer",
    "filter.overallStatus": "TERMINATED,WITHDRAWN,UNKNOWN,SUSPENDED,NO_LONGER_AVAILABLE",
    "fields": "NCTId,BriefTitle,OverallStatus,WhyStopped,StartDate,CompletionDate,LocationCountry,LeadSponsorName",
    "pageSize": 200
}

all_studies = []
while True:
    r = requests.get(base_url, params=params)
    data = r.json()
    all_studies.extend(data["studies"])
    print(f"Fetched {len(all_studies)} so far...")
    token = data.get("nextPageToken")
    if not token:
        break
    params["pageToken"] = token

with open("all_stalled_trials.json", "w") as f:
    json.dump({"studies": all_studies}, f)

print(f"Done. Total: {len(all_studies)}")

Fetched 200 so far...
Fetched 400 so far...
Fetched 600 so far...
Fetched 800 so far...
Fetched 954 so far...
Done. Total: 954


In [ ]:
print(f"Shape of the flattened DataFrame: {df_flattened.shape}")
print("Columns in the flattened DataFrame:")
for col in df_flattened.columns:
    print(f"- {col}")

print("\nValue counts for OverallStatus:")
display(df_flattened['OverallStatus'].value_counts())

print("\nValue counts for WhyStopped (top 10):")
display(df_flattened['WhyStopped'].value_counts().head(10))

NameError: name 'df_flattened' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def extract_protocol_info(protocol_section):
    nct_id = protocol_section.get('identificationModule', {}).get('nctId')
    brief_title = protocol_section.get('identificationModule', {}).get('briefTitle')
    overall_status = protocol_section.get('statusModule', {}).get('overallStatus')
    why_stopped = protocol_section.get('statusModule', {}).get('whyStopped')
    start_date = protocol_section.get('statusModule', {}).get('startDateStruct', {}).get('date')
    completion_date = protocol_section.get('statusModule', {}).get('primaryCompletionDateStruct', {}).get('date') \
                    or protocol_section.get('statusModule', {}).get('completionDateStruct', {}).get('date')

    location_country_list = protocol_section.get('locationModule', {}).get('locations')
    location_country = None
    if location_country_list:
        # Assuming one country for simplicity, or taking the first one found
        location_country = location_country_list[0].get('country')

    lead_sponsor_name = protocol_section.get('sponsorCollaboratorsModule', {}).get('leadSponsor', {}).get('leadSponsorName')

    return {
        'NCTId': nct_id,
        'BriefTitle': brief_title,
        'OverallStatus': overall_status,
        'WhyStopped': why_stopped,
        'StartDate': start_date,
        'CompletionDate': completion_date,
        'LocationCountry': location_country,
        'LeadSponsorName': lead_sponsor_name
    }

In [ ]:
rows = []

for study in data["studies"]:
    proto = study["protocolSection"]

    ident = proto.get("identificationModule", {})
    status = proto.get("statusModule", {})
    sponsor = proto.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {})
    locations = proto.get("contactsLocationsModule", {}).get("locations", [])

    countries = list(set(l["country"] for l in locations if "country" in l))

    rows.append({
        "NCTId": ident.get("nctId"),
        "BriefTitle": ident.get("briefTitle"),
        "OverallStatus": status.get("overallStatus"),
        "WhyStopped": status.get("whyStopped"),
        "StartDate": status.get("startDateStruct", {}).get("date"),
        "CompletionDate": status.get("completionDateStruct", {}).get("date"),
        "LeadSponsorName": sponsor.get("name"),
        "LocationCountries": "|".join(countries),
        "NumCountries": len(countries)
    })

df_clean = pd.DataFrame(rows)

print(df_clean.isnull().sum())
print(df_clean.head())

NCTId                  0
BriefTitle             0
OverallStatus          0
WhyStopped           511
StartDate             15
CompletionDate        60
LeadSponsorName        0
LocationCountries      0
NumCountries           0
dtype: int64
         NCTId                                         BriefTitle  \
0  NCT05781984  Role of Surgery in Management of Pineal Region...   
1  NCT04860817  A Study of Anti-CD7 CAR-T Cells in Pediatric a...   
2  NCT00179816  Tandem Peripheral Blood Stem Cell (PBSC) Rescu...   
3  NCT02654340  Biomarkers for Tuberous Sclerosis Complex (Bio...   
4  NCT03585686  A Combination of Vemurafenib, Cytarabine and 2...   

  OverallStatus                       WhyStopped   StartDate CompletionDate  \
0       UNKNOWN                             None     2023-04        2024-01   
1     WITHDRAWN  No proper participant is found.  2021-12-01     2023-11-01   
2       UNKNOWN                             None     1999-04        2012-09   
3    TERMINATED         Study n

In [ ]:
df_clean.to_csv("stalled_trials_clean.csv", index=False)

from google.colab import files
files.download("stalled_trials_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_clean.to_csv("stalled_trials_clean.csv", index=False)

from google.colab import files
files.download("stalled_trials_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm

# Load your data
df = pd.read_csv('stalled_trials_clean.csv')

# Create FundingRelated flag
funding_keywords = ['funding', 'grant', 'business priorities',
                    'sponsor decision', 'development program',
                    'funding unavailable', 'loss of funding',
                    'contract', 'funding accrual']

df['FundingRelated'] = df['WhyStopped'].str.lower().str.contains(
    '|'.join(funding_keywords), na=False).astype(int)

# Convert 'StartDate' to datetime and extract year
df['StartYear'] = pd.to_datetime(df['StartDate'], errors='coerce').dt.year

# Create Post2025 flag
df['Post2025'] = (df['StartYear'] >= 2025).astype(int)

# Create SponsorType
def classify_sponsor(name):
    if pd.isna(name): return 'Other'
    name = name.lower()
    if any(w in name for w in ['pharma','inc.','ltd','corp','bioscience','therapeutics']): return 'Pharma'
    if any(w in name for w in ['university','hospital','institute','college','center']): return 'Academic'
    return 'Other'

df['SponsorType'] = df['LeadSponsorName'].apply(classify_sponsor)

# Encode SponsorType
df = pd.get_dummies(df, columns=['SponsorType'], drop_first=True)

# Run logistic regression
X = df[['Post2025', 'NumCountries', 'SponsorType_Pharma', 'SponsorType_Other']].fillna(0).astype(int)
X = sm.add_constant(X)
y = df['FundingRelated']

model = sm.Logit(y, X).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.195659
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:         FundingRelated   No. Observations:                  954
Model:                          Logit   Df Residuals:                      949
Method:                           MLE   Df Model:                            4
Date:                Tue, 14 Apr 2026   Pseudo R-squ.:                 0.04814
Time:                        18:19:15   Log-Likelihood:                -186.66
converged:                       True   LL-Null:                       -196.10
Covariance Type:            nonrobust   LLR p-value:                 0.0008302
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -3.2290      0.216    -14.948      0.000      -3.652      -2.806
Post2

In [ ]:
print(f"Shape of the flattened DataFrame: {df_flattened.shape}")
print("Columns in the flattened DataFrame:")
for col in df_flattened.columns:
    print(f"- {col}")

print("\nValue counts for OverallStatus:")
display(df_flattened['OverallStatus'].value_counts())

print("\nValue counts for WhyStopped (top 10):")
display(df_flattened['WhyStopped'].value_counts().head(10))

Shape of the flattened DataFrame: (954, 8)
Columns in the flattened DataFrame:
- NCTId
- BriefTitle
- OverallStatus
- WhyStopped
- StartDate
- CompletionDate
- LocationCountry
- LeadSponsorName

Value counts for OverallStatus:


,count
OverallStatus,
UNKNOWN,464
TERMINATED,326
WITHDRAWN,138
SUSPENDED,17
NO_LONGER_AVAILABLE,9



Value counts for WhyStopped (top 10):


,count
WhyStopped,
Slow accrual,9
Poor accrual,5
slow accrual,4
Lack of enrollment,4
Insufficient recruitment,3
Lack of accrual,3
low accrual,3
No participants enrolled,3
Slow accrual.,3


In [ ]:
print(f"Shape of the flattened DataFrame: {df_flattened.shape}")
print("Columns in the flattened DataFrame:")
for col in df_flattened.columns:
    print(f"- {col}")

print("\nValue counts for OverallStatus:")
display(df_flattened['OverallStatus'].value_counts())

print("\nValue counts for WhyStopped (top 10):")
display(df_flattened['WhyStopped'].value_counts().head(10))

Shape of the flattened DataFrame: (954, 8)
Columns in the flattened DataFrame:
- NCTId
- BriefTitle
- OverallStatus
- WhyStopped
- StartDate
- CompletionDate
- LocationCountry
- LeadSponsorName

Value counts for OverallStatus:


,count
OverallStatus,
UNKNOWN,464
TERMINATED,326
WITHDRAWN,138
SUSPENDED,17
NO_LONGER_AVAILABLE,9



Value counts for WhyStopped (top 10):


,count
WhyStopped,
Slow accrual,9
Poor accrual,5
slow accrual,4
Lack of enrollment,4
Insufficient recruitment,3
Lack of accrual,3
low accrual,3
No participants enrolled,3
Slow accrual.,3


In [ ]:
import pandas as pd
import json

with open("all_stalled_trials.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data["studies"])
display(df.head())

,protocolSection
0,{'identificationModule': {'nctId': 'NCT0578198...
1,{'identificationModule': {'nctId': 'NCT0486081...
2,{'identificationModule': {'nctId': 'NCT0017981...
3,{'identificationModule': {'nctId': 'NCT0265434...
4,{'identificationModule': {'nctId': 'NCT0358568...


In [ ]:
def extract_protocol_info(protocol_section):
    nct_id = protocol_section.get('identificationModule', {}).get('nctId')
    brief_title = protocol_section.get('identificationModule', {}).get('briefTitle')
    overall_status = protocol_section.get('statusModule', {}).get('overallStatus')
    why_stopped = protocol_section.get('statusModule', {}).get('whyStopped')
    start_date = protocol_section.get('statusModule', {}).get('startDateStruct', {}).get('date')
    completion_date = protocol_section.get('statusModule', {}).get('primaryCompletionDateStruct', {}).get('date') \
                    or protocol_section.get('statusModule', {}).get('completionDateStruct', {}).get('date')

    location_country_list = protocol_section.get('locationModule', {}).get('locations')
    location_country = None
    if location_country_list:
        # Assuming one country for simplicity, or taking the first one found
        location_country = location_country_list[0].get('country')

    lead_sponsor_name = protocol_section.get('sponsorCollaboratorsModule', {}).get('leadSponsor', {}).get('leadSponsorName')

    return {
        'NCTId': nct_id,
        'BriefTitle': brief_title,
        'OverallStatus': overall_status,
        'WhyStopped': why_stopped,
        'StartDate': start_date,
        'CompletionDate': completion_date,
        'LocationCountry': location_country,
        'LeadSponsorName': lead_sponsor_name
    }

In [ ]:
# Apply the function to the 'protocolSection' column
flattened_data = df['protocolSection'].apply(extract_protocol_info)

# Convert the list of dictionaries into a DataFrame
flattened_df = pd.DataFrame(flattened_data.tolist())

# Drop the original 'protocolSection' column and concatenate with the new flattened DataFrame
df_flattened = pd.concat([df.drop(columns=['protocolSection']), flattened_df], axis=1)

display(df_flattened.head())

,NCTId,BriefTitle,OverallStatus,WhyStopped,StartDate,CompletionDate,LocationCountry,LeadSponsorName
0,NCT05781984,Role of Surgery in Management of Pineal Region...,UNKNOWN,None,2023-04,2024-01,None,None
1,NCT04860817,A Study of Anti-CD7 CAR-T Cells in Pediatric a...,WITHDRAWN,No proper participant is found.,2021-12-01,2023-11-01,None,None
2,NCT00179816,Tandem Peripheral Blood Stem Cell (PBSC) Rescu...,UNKNOWN,None,1999-04,2012-09,None,None
3,NCT02654340,Biomarkers for Tuberous Sclerosis Complex (Bio...,TERMINATED,Study no longer pursued.,2018-08-01,2022-12-30,None,None
4,NCT03585686,"A Combination of Vemurafenib, Cytarabine and 2...",UNKNOWN,None,2018-06-26,2023-12,None,None


In [ ]:
# The original 'df' DataFrame created earlier appears to have already flattened
# the data from 'protocolSection' into top-level columns, hence the 'protocolSection'
# column no longer exists in 'df', leading to the KeyError.

# To correctly use the 'extract_protocol_info' function, we need to apply it
# to the 'protocolSection' dictionary of each study from the raw 'data'.

# Apply the function directly to the 'protocolSection' of each study in the raw data
flattened_data_list = [extract_protocol_info(study['protocolSection']) for study in data['studies']]

# Convert the list of dictionaries into a DataFrame
df_flattened = pd.DataFrame(flattened_data_list)

display(df_flattened.head())

,NCTId,BriefTitle,OverallStatus,WhyStopped,StartDate,CompletionDate,LocationCountry,LeadSponsorName
0,NCT05781984,Role of Surgery in Management of Pineal Region...,UNKNOWN,None,2023-04,2024-01,None,None
1,NCT04860817,A Study of Anti-CD7 CAR-T Cells in Pediatric a...,WITHDRAWN,No proper participant is found.,2021-12-01,2023-11-01,None,None
2,NCT00179816,Tandem Peripheral Blood Stem Cell (PBSC) Rescu...,UNKNOWN,None,1999-04,2012-09,None,None
3,NCT02654340,Biomarkers for Tuberous Sclerosis Complex (Bio...,TERMINATED,Study no longer pursued.,2018-08-01,2022-12-30,None,None
4,NCT03585686,"A Combination of Vemurafenib, Cytarabine and 2...",UNKNOWN,None,2018-06-26,2023-12,None,None


In [ ]:
import pandas as pd
import json

with open("all_stalled_trials.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data["studies"])
display(df.head())

,protocolSection
0,{'identificationModule': {'nctId': 'NCT0578198...
1,{'identificationModule': {'nctId': 'NCT0486081...
2,{'identificationModule': {'nctId': 'NCT0017981...
3,{'identificationModule': {'nctId': 'NCT0265434...
4,{'identificationModule': {'nctId': 'NCT0358568...


In [ ]:
print(df_flattened.isnull().sum())
print(f"\)nTotal rows: {len(df_flattened)}")

NCTId                0
BriefTitle           0
OverallStatus        0
WhyStopped         511
StartDate           15
CompletionDate      60
LocationCountry    954
LeadSponsorName    954
dtype: int64
\)nTotal rows: 954


<>:2: SyntaxWarning: invalid escape sequence '\)'
<>:2: SyntaxWarning: invalid escape sequence '\)'
/tmp/ipykernel_5491/3814657822.py:2: SyntaxWarning: invalid escape sequence '\)'
  print(f"\)nTotal rows: {len(df_flattened)}")


In [ ]:
rows = []

for study in data["studies"]:
    proto = study["protocolSection"]

    ident = proto.get("identificationModule", {})
    status = proto.get("statusModule", {})
    sponsor = proto.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {})
    locations = proto.get("contactsLocationsModule", {}).get("locations", [])

    countries = list(set(l["country"] for l in locations if "country" in l))

    rows.append({
        "NCTId": ident.get("nctId"),
        "BriefTitle": ident.get("briefTitle"),
        "OverallStatus": status.get("overallStatus"),
        "WhyStopped": status.get("whyStopped"),
        "StartDate": status.get("startDateStruct", {}).get("date"),
        "CompletionDate": status.get("completionDateStruct", {}).get("date"),
        "LeadSponsorName": sponsor.get("name"),
        "LocationCountries": "|".join(countries),
        "NumCountries": len(countries)
    })

df_clean = pd.DataFrame(rows)

print(df_clean.isnull().sum())
print(df_clean.head())

NCTId                  0
BriefTitle             0
OverallStatus          0
WhyStopped           511
StartDate             15
CompletionDate        60
LeadSponsorName        0
LocationCountries      0
NumCountries           0
dtype: int64
         NCTId                                         BriefTitle  \
0  NCT05781984  Role of Surgery in Management of Pineal Region...   
1  NCT04860817  A Study of Anti-CD7 CAR-T Cells in Pediatric a...   
2  NCT00179816  Tandem Peripheral Blood Stem Cell (PBSC) Rescu...   
3  NCT02654340  Biomarkers for Tuberous Sclerosis Complex (Bio...   
4  NCT03585686  A Combination of Vemurafenib, Cytarabine and 2...   

  OverallStatus                       WhyStopped   StartDate CompletionDate  \
0       UNKNOWN                             None     2023-04        2024-01   
1     WITHDRAWN  No proper participant is found.  2021-12-01     2023-11-01   
2       UNKNOWN                             None     1999-04        2012-09   
3    TERMINATED         Study n

In [ ]:
df_clean.to_csv("stalled_trials_clean.csv", index=False)

from google.colab import files
files.download("stalled_trials_clean.csv")

NameError: name 'df_clean' is not defined

In [ ]:
from google.collab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm

# Load your data
df = pd.read_csv('stalled_trials_clean.csv')

# Create FundingRelated flag
funding_keywords = ['funding', 'grant', 'business priorities',
                    'sponsor decision', 'development program',
                    'funding unavailable', 'loss of funding',
                    'contract', 'funding accrual']

df['FundingRelated'] = df['WhyStopped'].str.lower().str.contains(
    '|'.join(funding_keywords), na=False).astype(int)

# Convert 'StartDate' to datetime and extract year
df['StartYear'] = pd.to_datetime(df['StartDate'], errors='coerce').dt.year

# Create Post2025 flag
df['Post2025'] = (df['StartYear'] >= 2025).astype(int)

# Create SponsorType
def classify_sponsor(name):
    if pd.isna(name): return 'Other'
    name = name.lower()
    if any(w in name for w in ['pharma','inc.','ltd','corp','bioscience','therapeutics']): return 'Pharma'
    if any(w in name for w in ['university','hospital','institute','college','center']): return 'Academic'
    return 'Other'

df['SponsorType'] = df['LeadSponsorName'].apply(classify_sponsor)

# Encode SponsorType
df = pd.get_dummies(df, columns=['SponsorType'], drop_first=True)

# Run logistic regression
X = df[['Post2025', 'NumCountries', 'SponsorType_Pharma', 'SponsorType_Other']].fillna(0)
X = sm.add_constant(X)
X = X.astype(float) # Ensure all columns in X are float type
y = df['FundingRelated']

model = sm.Logit(y, X).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.195659
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:         FundingRelated   No. Observations:                  954
Model:                          Logit   Df Residuals:                      949
Method:                           MLE   Df Model:                            4
Date:                Tue, 14 Apr 2026   Pseudo R-squ.:                 0.04814
Time:                        18:26:15   Log-Likelihood:                -186.66
converged:                       True   LL-Null:                       -196.10
Covariance Type:            nonrobust   LLR p-value:                 0.0008302
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -3.2290      0.216    -14.948      0.000      -3.652      -2.806
Post2

In [ ]:
df['WhyStopped'] = df['WhyStopped'].str.lower()
df['FundingRelated'] = df['WhyStopped'].str.contains('|'.join(funding_keywords) , na=False).astype(int)